# Align a molecule with PCA

This notebook shows a simple way to **rotate a molecule so that its main geometric axis points along the z-axis**.

The idea is:

1. load the molecule with **ASE**,
2. identify its main geometric direction with **PCA**,
3. compute a rotation that aligns that direction with the **z-axis**,
4. visualize the structure before and after the rotation.

This is useful when you want a cleaner orientation for visualization, comparison between structures, or further analysis.

## 1. Import the libraries

We use:

- **NumPy** for arrays and vector operations,
- **scikit-learn** for PCA,
- **SciPy** for the rotation,
- **ASE** to read and manipulate atomic structures,
- **nglview** to visualize the molecule inside the notebook.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from scipy.spatial.transform import Rotation as R
from ase.io import read, write
import nglview as nv

## 2. A small visualization helper

This function creates an interactive 3D view of an ASE structure.

A few notes:

- atom indices are shown, which is useful for teaching and debugging;
- `highlight_selection` can be used to emphasize selected atoms;
- the default is `None` rather than an empty list, because mutable default arguments are best avoided.

In [ ]:
def view_structure(structure, highlight_selection=None):
    """Return an NGLView widget for an ASE structure.

    Parameters
    ----------
    structure : ase.Atoms
        Structure to display.
    highlight_selection : str or None
        Optional NGL selection string used to highlight atoms.
        Example: "0 1 2" to highlight atoms with indices 0, 1, and 2.
    """
    widget = nv.NGLWidget(nv.ASEStructure(structure), gui=True)
    widget.add_unitcell()
    widget.add_ball_and_stick()
    widget.add_representation("label", label_type="atomindex", color="black")

    if highlight_selection is not None:
        widget.add_representation(
            "spacefill",
            selection=highlight_selection,
            color="blue",
            radius=0.5,
        )

    return widget

## 3. Load the molecule

Here we read the structure from the file `unde.xyz`.

We keep a copy of the original structure, so that we can later compare the initial and rotated geometries.

In [ ]:
molecule = read("unde.xyz")
original_molecule = molecule.copy()

### Original structure

In [ ]:
view_structure(original_molecule)

## 4. Find the main molecular axis with PCA

PCA (**Principal Component Analysis**) finds directions in space along which the atomic positions vary the most.

For a molecule:

- the **first principal component** is often a good estimate of its longest geometric direction;
- the **second** and **third** components are orthogonal directions with smaller spread.

Here we use the **first principal axis** as the direction we want to align with the z-axis.

### Important geometric note

The sign of a PCA axis is arbitrary: if a vector is a principal axis, then its negative is also a valid principal axis.

So aligning the molecule with `+z` or `-z` differs only by a 180° flip, and both are geometrically acceptable unless you need a specific orientation.

In [ ]:
# Extract Cartesian coordinates of all atoms
coordinates = original_molecule.get_positions()

# Fit PCA to the atomic coordinates
pca = PCA(n_components=3)
pca.fit(coordinates)

# Each row of pca.components_ is a principal direction
principal_axes = pca.components_

# The first principal axis corresponds to the largest spatial spread
principal_axis = principal_axes[0]

# We want to align that axis with the z-direction
target_axis = np.array([0.0, 0.0, 1.0])

print("First principal axis (before alignment):", principal_axis)
print("Target axis:", target_axis)

## 5. Compute and apply the rotation

`Rotation.align_vectors` finds the rotation that best maps one set of vectors onto another.

Here we use it with a single vector:

- source: the molecular principal axis,
- target: the z-axis.

Then we apply the rotation to all atomic coordinates.

In [ ]:
# Compute the rotation that aligns the principal axis with the z-axis
rotation, rmsd = R.align_vectors([target_axis], [principal_axis])

# Apply the rotation to a copy of the molecule
aligned_molecule = original_molecule.copy()
rotated_coordinates = rotation.apply(coordinates)
aligned_molecule.set_positions(rotated_coordinates)

print(f"RMSD returned by align_vectors: {rmsd:.6f}")

### Rotated structure

In [ ]:
view_structure(aligned_molecule)

## 6. Verify that the alignment worked

A simple check is to run PCA again on the rotated molecule.

If everything worked as expected, the new first principal axis should be very close to the z-axis (or to `-z`, because of the sign ambiguity mentioned above).

In [ ]:
rotated_pca = PCA(n_components=3)
rotated_pca.fit(aligned_molecule.get_positions())

rotated_principal_axis = rotated_pca.components_[0]

# Compute the angle between the new principal axis and the z-axis
cos_angle = np.dot(rotated_principal_axis, target_axis)
cos_angle = np.clip(cos_angle, -1.0, 1.0)
angle_deg = np.degrees(np.arccos(abs(cos_angle)))  # abs() ignores the ± sign ambiguity

print("First principal axis (after alignment):", rotated_principal_axis)
print(f"Angle with the z-axis (ignoring sign): {angle_deg:.6f} degrees")

## 7. Optional: save the aligned structure

Uncomment the next cell if you want to write the aligned structure to a new XYZ file.

In [ ]:
# write("unde_aligned.xyz", aligned_molecule)

## Summary

In this notebook you have seen how to:

- load a molecular structure with ASE,
- use PCA to identify its main geometric axis,
- rotate the structure so that this axis points along z,
- check that the transformation worked.

This is a good example of how a simple statistical tool such as PCA can be used for a concrete structural manipulation task in atomistic modeling.